[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_12/NB_bishop_ch12_transformers.ipynb)

# Chapter 12 -- Transformers

This notebook implements the **Transformer architecture** from scratch:
- Scaled dot-product attention
- Multi-head attention
- Positional encoding
- Full Transformer encoder
- BERT-style masked language modeling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}, Device: {device}')

## 1. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention_numpy(Q, K, V, mask=None):
    """NumPy implementation of scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    return weights @ V, weights

np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention_numpy(Q, K, V)
print(f'Output shape: {output.shape}')
print(f'Attention weights (rows sum to 1):')
print(weights.round(3))

In [ ]:
tokens = ['The', 'cat', 'sat', 'down']
causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
_, weights_causal = scaled_dot_product_attention_numpy(Q, K, V, mask=causal_mask)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(weights, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(len(tokens))); ax1.set_yticks(range(len(tokens)))
ax1.set_xticklabels(tokens); ax1.set_yticklabels(tokens)
ax1.set_xlabel('Key'); ax1.set_ylabel('Query')
ax1.set_title('Self-Attention Weights', fontweight='bold')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax1.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=10)

im2 = ax2.imshow(weights_causal, cmap='Reds', aspect='auto', vmin=0, vmax=1)
ax2.set_xticks(range(len(tokens))); ax2.set_yticks(range(len(tokens)))
ax2.set_xticklabels(tokens); ax2.set_yticklabels(tokens)
ax2.set_xlabel('Key'); ax2.set_ylabel('Query')
ax2.set_title('Causal (Masked) Attention', fontweight='bold')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax2.text(j, i, f'{weights_causal[i,j]:.2f}', ha='center', va='center', fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_attention')
plt.show()

## 2. Multi-Head Attention (PyTorch)

In [ ]:
def scaled_dot_product_attention_torch(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        out, attn = scaled_dot_product_attention_torch(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(out), attn

d_model, n_heads = 64, 8
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)
output, attn_w = mha(x, x, x)
print(f'd_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}')
print(f'Output shape: {output.shape}, Attention shape: {attn_w.shape}')
print(f'Parameters: {sum(p.numel() for p in mha.parameters()):,}')

In [ ]:
# Visualize attention per head
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for h in range(8):
    r, c = h // 4, h % 4
    axes[r, c].imshow(attn_w[0, h].detach().numpy(), cmap='Blues', aspect='auto')
    axes[r, c].set_title(f'Head {h+1}', fontweight='bold')
    axes[r, c].set_xlabel('Key')
    if c == 0:
        axes[r, c].set_ylabel('Query')
plt.suptitle('Multi-Head Attention Patterns (8 Heads)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'bishop_ch12_multihead')
plt.show()

## 3. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

d_model_vis = 128
pe_module = PositionalEncoding(d_model_vis, 100, dropout=0.0)
pe_values = pe_module.pe.squeeze(0).numpy()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im = ax1.imshow(pe_values[:50, :64], cmap='RdBu_r', aspect='auto', interpolation='nearest')
ax1.set_xlabel('Encoding Dimension'); ax1.set_ylabel('Position')
ax1.set_title('Positional Encoding Heatmap', fontweight='bold')
plt.colorbar(im, ax=ax1, shrink=0.8)

dims_to_plot = [0, 1, 4, 5, 20, 21]
colors_pe = ['#1a3a6e', '#cd0000', '#228B22', '#DAA520', '#8B008B', '#FF6347']
for dim, color in zip(dims_to_plot, colors_pe):
    ax2.plot(pe_values[:50, dim], color=color, linewidth=1.5,
             label=f'dim {dim} ({"sin" if dim % 2 == 0 else "cos"})')
ax2.set_xlabel('Position'); ax2.set_ylabel('Value')
ax2.set_title('Positional Encoding by Dimension', fontweight='bold')
ax2.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_positional')
plt.show()

## 4. Positional Encoding Distance Pattern

In [ ]:
pe_50 = pe_values[:50]
similarity = pe_50 @ pe_50.T

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(similarity, cmap='viridis', aspect='auto')
ax.set_xlabel('Position'); ax.set_ylabel('Position')
ax.set_title('PE Dot-Product Similarity (relative distance pattern)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

## 5. Transformer Encoder Block

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_w = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ff(x)))
        return x, attn_w

block = TransformerEncoderBlock(d_model=64, n_heads=8, d_ff=256)
out, attn = block(torch.randn(2, 10, 64))
print(f'Encoder block output: {out.shape}')
print(f'Parameters: {sum(p.numel() for p in block.parameters()):,}')

## 6. Full Transformer Encoder

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)])
        self.d_model = d_model

    def forward(self, x, mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        all_attn = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            all_attn.append(attn_w)
        return x, all_attn

encoder = TransformerEncoder(vocab_size=10000, d_model=64, n_heads=8, d_ff=256, n_layers=4)
tokens_enc = torch.randint(0, 10000, (2, 20))
out, all_attn = encoder(tokens_enc)
print(f'Encoder output: {out.shape}, Layers: {len(all_attn)}')
print(f'Total parameters: {sum(p.numel() for p in encoder.parameters()):,}')

In [ ]:
# Visualize attention across layers
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Attention: Layer 0 (top) vs Layer 3 (bottom), Heads 0-3', fontsize=14)

for col in range(4):
    attn_l0 = all_attn[0][0, col, :10, :10].detach().numpy()
    axes[0, col].imshow(attn_l0, cmap='Blues', aspect='auto', vmin=0)
    axes[0, col].set_title(f'L0 Head {col}')
    attn_l3 = all_attn[3][0, col, :10, :10].detach().numpy()
    axes[1, col].imshow(attn_l3, cmap='Reds', aspect='auto', vmin=0)
    axes[1, col].set_title(f'L3 Head {col}')

fig.tight_layout()
save_fig(fig, 'bishop_ch12_transformer_block')
plt.show()

## 7. BERT-Style Masked Language Modeling

In [ ]:
class SimpleBERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=256, n_layers=2):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, n_heads, d_ff, n_layers)
        self.mlm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        enc_out, attns = self.encoder(x, mask)
        return self.mlm_head(enc_out), attns

vocab_size_bert = 100
MASK_TOKEN = 0
bert = SimpleBERT(vocab_size=vocab_size_bert, d_model=64, n_heads=4)
print(f'SimpleBERT parameters: {sum(p.numel() for p in bert.parameters()):,}')

In [ ]:
optimizer_bert = torch.optim.Adam(bert.parameters(), lr=1e-3)
losses_bert = []

for epoch in range(50):
    batch = torch.randint(1, vocab_size_bert, (32, 20))
    targets = batch.clone()
    mask_indices = torch.rand(batch.shape) < 0.15
    batch[mask_indices] = MASK_TOKEN
    logits, _ = bert(batch)
    loss = F.cross_entropy(logits[mask_indices], targets[mask_indices])
    optimizer_bert.zero_grad()
    loss.backward()
    optimizer_bert.step()
    losses_bert.append(loss.item())

print(f'Final MLM loss: {losses_bert[-1]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses_bert, color='#1a3a6e', lw=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MLM Loss')
ax.set_title('BERT Masked Language Model Training', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'bishop_ch12_bert')
plt.show()

## 8. BERT Fine-Tuning with Hugging Face

In [ ]:
# !pip install transformers datasets -q
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset

dataset = load_dataset('imdb', split={'train': 'train[:500]', 'test': 'test[:200]'})
print(f'Train: {len(dataset["train"])}, Test: {len(dataset["test"])}')

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print(f'Tokenized keys: {list(tokenized["train"][0].keys())}')

In [ ]:
model_bert = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./bert_imdb', num_train_epochs=2,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    warmup_steps=50, weight_decay=0.01, logging_steps=25,
    eval_strategy='epoch', save_strategy='no', report_to='none',
    fp16=torch.cuda.is_available())

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'accuracy': (np.argmax(logits, axis=-1) == labels).mean()}

trainer = Trainer(model=model_bert, args=training_args,
                  train_dataset=tokenized['train'], eval_dataset=tokenized['test'],
                  compute_metrics=compute_metrics)

print(f'BERT parameters: {model_bert.num_parameters():,}')
print('Starting fine-tuning...')

In [ ]:
train_result = trainer.train()
eval_result = trainer.evaluate()
print(f'Training loss: {train_result.training_loss:.4f}')
print(f'Eval accuracy: {eval_result["eval_accuracy"]:.4f}')

In [ ]:
log_history = trainer.state.log_history
train_losses_hf = [(h['step'], h['loss']) for h in log_history if 'loss' in h]
eval_accs = [(h['step'], h['eval_accuracy']) for h in log_history if 'eval_accuracy' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
if train_losses_hf:
    steps, losses = zip(*train_losses_hf)
    ax1.plot(steps, losses, 'o-', color='#1a3a6e', linewidth=2)
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('BERT Fine-Tuning Loss', fontweight='bold')

if eval_accs:
    steps_e, accs_e = zip(*eval_accs)
    ax2.bar(range(len(accs_e)), accs_e, color='#228B22', alpha=0.8)
    ax2.set_xticks(range(len(accs_e)))
    ax2.set_xticklabels([f'Epoch {i+1}' for i in range(len(accs_e))])
    ax2.set_ylabel('Accuracy'); ax2.set_title('BERT Eval Accuracy', fontweight='bold')
    ax2.set_ylim(0, 1)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_bert_finetune')
plt.show()

## 9. Transformer for Classification

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, d_model=64, n_heads=4, d_ff=128, n_layers=2):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, n_heads, d_ff, n_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        return self.classifier(enc_out[:, 0])

clf = TransformerClassifier(vocab_size=100, n_classes=3)
clf_opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
clf_losses = []

for epoch in range(80):
    batch_clf = torch.randint(1, 100, (32, 15))
    labels_clf = (batch_clf[:, :5].float().mean(dim=1) % 3).long()
    logits_clf = clf(batch_clf)
    loss_clf = F.cross_entropy(logits_clf, labels_clf)
    clf_opt.zero_grad(); loss_clf.backward(); clf_opt.step()
    clf_losses.append(loss_clf.item())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(clf_losses, color='#cd0000', lw=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Transformer Classifier Training', fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

- **Scaled dot-product attention** computes weighted sums based on query-key similarity
- **Multi-head attention** attends to different representation subspaces
- **Positional encoding** injects sequence order via sinusoidal functions
- **BERT** uses masked language modeling for bidirectional pre-training
- Transformers are the dominant architecture in NLP and increasingly in vision/science

In [ ]:
takeaways = [
    '1. Attention computes a weighted sum of values, with weights from query-key similarity.',
    '2. Scaling by sqrt(d_k) prevents softmax saturation for large dimensions.',
    '3. Multi-head attention lets the model attend to different representation subspaces.',
    '4. Positional encoding is essential since attention is permutation-invariant.',
    '5. BERT fine-tuning achieves strong results even with very small datasets.'
]
print('Key Takeaways:')
for t in takeaways:
    print(t)